# Collaborative Filtering — Recommendation Systems

- **User-Based CF**: Find similar users → recommend what they liked
- **Item-Based CF**: Find similar items → recommend related items
- **Matrix Factorization**: SVD-based latent factor models
- **Evaluation**: RMSE, Precision@K, Recall@K

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.sparse.linalg import svds
import warnings
warnings.filterwarnings('ignore')

## 1. Create Synthetic Movie Ratings Dataset

In [ ]:
# Generate synthetic ratings data
np.random.seed(42)
n_users, n_items = 200, 50
movies = [f'Movie_{i}' for i in range(1, n_items + 1)]
users = [f'User_{i}' for i in range(1, n_users + 1)]

# Create sparse ratings (not everyone rates every movie)
ratings_list = []
for user_id in range(n_users):
    # Each user rates 10-30 movies
    n_rated = np.random.randint(10, 30)
    rated_items = np.random.choice(n_items, n_rated, replace=False)
    for item_id in rated_items:
        rating = np.clip(np.random.normal(3.5, 1.2), 1, 5)
        ratings_list.append({'userId': user_id, 'movieId': item_id, 'rating': round(rating, 1)})

ratings_df = pd.DataFrame(ratings_list)
print(f'Dataset: {len(ratings_df)} ratings | {n_users} users | {n_items} movies')
print(f'Sparsity: {1 - len(ratings_df) / (n_users * n_items):.2%}')
print(f'\nRating Distribution:')
print(ratings_df['rating'].describe())

In [ ]:
# Create User-Item Matrix
user_item_matrix = ratings_df.pivot_table(index='userId', columns='movieId', values='rating')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(user_item_matrix.iloc[:20, :20], cmap='YlOrRd', ax=axes[0], 
            cbar_kws={'label': 'Rating'}, mask=user_item_matrix.iloc[:20, :20].isna())
axes[0].set_title('User-Item Rating Matrix (20x20 subset)')

axes[1].hist(ratings_df['rating'], bins=20, color='steelblue', edgecolor='black')
axes[1].set_xlabel('Rating'); axes[1].set_ylabel('Count')
axes[1].set_title('Rating Distribution')
axes[1].axvline(ratings_df['rating'].mean(), color='red', linestyle='--', label=f'Mean: {ratings_df["rating"].mean():.2f}')
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. User-Based Collaborative Filtering

In [ ]:
# Fill NaN with 0 for similarity computation
user_item_filled = user_item_matrix.fillna(0)

# Compute User-User Cosine Similarity
user_similarity = cosine_similarity(user_item_filled)
user_sim_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

plt.figure(figsize=(8, 6))
sns.heatmap(user_sim_df.iloc[:20, :20], cmap='coolwarm', center=0, vmin=-0.2, vmax=1)
plt.title('User-User Similarity Matrix (Top 20)')
plt.tight_layout()
plt.show()

In [ ]:
def predict_user_based(user_id, item_id, user_item_matrix, user_similarity, k=10):
    """Predict rating using k most similar users who rated the item."""
    if item_id not in user_item_matrix.columns:
        return user_item_matrix.mean().mean()  # global mean fallback
    
    # Get users who rated this item
    item_ratings = user_item_matrix[item_id].dropna()
    if len(item_ratings) == 0:
        return user_item_matrix.mean().mean()
    
    # Get similarity scores for these users
    sim_scores = user_similarity[user_id][item_ratings.index]
    
    # Top-k most similar users
    top_k_idx = np.argsort(sim_scores)[::-1][:k]
    top_k_sims = sim_scores[top_k_idx]
    top_k_ratings = item_ratings.iloc[top_k_idx]
    
    if np.sum(np.abs(top_k_sims)) == 0:
        return user_item_matrix.mean().mean()
    
    return np.dot(top_k_sims, top_k_ratings) / np.sum(np.abs(top_k_sims))

# Recommend for user 0
user_id = 0
unrated_items = user_item_matrix.columns[user_item_matrix.loc[user_id].isna()]
predictions = {item: predict_user_based(user_id, item, user_item_matrix, user_similarity)
               for item in unrated_items}
top_recs = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:10]
print(f'Top 10 Recommendations for User {user_id}:')
for movie_id, pred_rating in top_recs:
    print(f'  Movie_{movie_id+1}: {pred_rating:.2f}')

## 3. Item-Based Collaborative Filtering

In [ ]:
# Item-Item Similarity
item_similarity = cosine_similarity(user_item_filled.T)
item_sim_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

def predict_item_based(user_id, item_id, user_item_matrix, item_similarity, k=10):
    """Predict rating using k most similar items that the user rated."""
    user_ratings = user_item_matrix.loc[user_id].dropna()
    if len(user_ratings) == 0:
        return user_item_matrix.mean().mean()
    
    # Get similarity between target item and items user has rated
    item_idx = list(user_item_matrix.columns).index(item_id)
    rated_item_indices = [list(user_item_matrix.columns).index(i) for i in user_ratings.index]
    sim_scores = item_similarity[item_idx][rated_item_indices]
    
    top_k_idx = np.argsort(sim_scores)[::-1][:k]
    top_k_sims = sim_scores[top_k_idx]
    top_k_ratings = user_ratings.values[top_k_idx]
    
    if np.sum(np.abs(top_k_sims)) == 0:
        return user_item_matrix.mean().mean()
    
    return np.dot(top_k_sims, top_k_ratings) / np.sum(np.abs(top_k_sims))

# Recommend for user 0 using item-based CF
predictions_item = {item: predict_item_based(user_id, item, user_item_matrix, item_similarity)
                    for item in unrated_items}
top_recs_item = sorted(predictions_item.items(), key=lambda x: x[1], reverse=True)[:10]
print(f'Item-Based Top 10 for User {user_id}:')
for movie_id, pred_rating in top_recs_item:
    print(f'  Movie_{movie_id+1}: {pred_rating:.2f}')

## 4. Matrix Factorization (SVD)

In [ ]:
# SVD-based Matrix Factorization
# Fill NaN with user mean
user_means = user_item_matrix.mean(axis=1)
matrix_centered = user_item_matrix.sub(user_means, axis=0).fillna(0)

# Perform SVD (k latent factors)
k = 15
U, sigma, Vt = svds(matrix_centered.values.astype(float), k=k)
sigma_diag = np.diag(sigma)

# Reconstruct predictions
predicted_ratings = np.dot(np.dot(U, sigma_diag), Vt) + user_means.values.reshape(-1, 1)
predicted_df = pd.DataFrame(predicted_ratings, index=user_item_matrix.index, columns=user_item_matrix.columns)

# Recommendations from SVD
svd_preds = {item: predicted_df.loc[user_id, item] for item in unrated_items}
top_svd = sorted(svd_preds.items(), key=lambda x: x[1], reverse=True)[:10]
print(f'SVD Top 10 for User {user_id}:')
for movie_id, pred_rating in top_svd:
    print(f'  Movie_{movie_id+1}: {pred_rating:.2f}')

In [ ]:
# Evaluate all methods on held-out ratings
from sklearn.metrics import mean_squared_error

# Get actual ratings for evaluation
test_ratings = ratings_df.sample(frac=0.2, random_state=42)

# Predict with each method
rmse_results = {}
for method_name, pred_func in [('User-Based', lambda u, i: predict_user_based(u, i, user_item_matrix, user_similarity)),
                                ('Item-Based', lambda u, i: predict_item_based(u, i, user_item_matrix, item_similarity)),
                                ('SVD', lambda u, i: predicted_df.loc[u, i] if u in predicted_df.index and i in predicted_df.columns else 3.5)]:
    preds = [pred_func(row['userId'], row['movieId']) for _, row in test_ratings.iterrows()]
    rmse = np.sqrt(mean_squared_error(test_ratings['rating'], preds))
    rmse_results[method_name] = rmse

# Plot RMSE comparison
plt.figure(figsize=(8, 5))
colors = ['#3498db', '#e67e22', '#2ecc71']
bars = plt.bar(rmse_results.keys(), rmse_results.values(), color=colors, edgecolor='black')
for bar, val in zip(bars, rmse_results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.3f}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.title('RMSE Comparison — Collaborative Filtering Methods')
plt.ylabel('RMSE (lower is better)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Summary

| Method | Pros | Cons |
|--------|------|------|
| **User-Based CF** | Intuitive, serendipitous recs | Scalability, cold start |
| **Item-Based CF** | Stable, scalable (items change less) | Less diverse recs |
| **SVD/MF** | Handles sparsity well, captures latent factors | Less interpretable |
| **Deep Learning (NCF)** | Non-linear, flexible | Needs more data |

### Cold Start Problem
- **New user**: No ratings → Use content-based or popularity-based
- **New item**: No ratings → Use content-based features
- **Solution**: Hybrid approach (collaborative + content-based)